This may have to exist outside Round 1 as it is univariate.

In [ ]:
%%time

# Import libraries
import os
import joblib

import numpy as np
import pandas as pd

from prophet import Prophet

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# Load dataset
df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)


# Prepare dataset
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# Next day closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker for which next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# Next close change
df_model['next_close_change'] = (
    df_model['next_close'] - df_model['close']
)

# Next close change PCT - This will be the target
df_model['next_close_change_pct'] = (
    (df_model['next_close'] - df_model['close']) / df_model['close']
) * 100


# Train/test split
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# Prophet predictions container
all_results = []

tickers = sorted(test_df['ticker'].unique())

for ticker in tickers:
    ticker_train = train_df[train_df['ticker'] == ticker].copy()
    ticker_test = test_df[test_df['ticker'] == ticker].copy()

    # Prophet requires columns named ds and y
    prophet_train = ticker_train.rename(columns={
        'timestamp': 'ds',
        'next_close_change_pct': 'y'
    })[['ds', 'y']]

    prophet_future = ticker_test.rename(columns={
        'timestamp': 'ds'
    })[['ds']]

    # Skip ticker if not enough training data
    if len(prophet_train) < 30:
        continue

    # Define model
    model = Prophet()

    # Train model
    model.fit(prophet_train)

    # Predict 2025 dates
    forecast = model.predict(prophet_future)

    ticker_results = ticker_test.copy()
    ticker_results['Prophet_pred_next_close_change_pct'] = forecast['yhat'].values

    all_results.append(ticker_results)




# Get predictions
y_test_pred = results_df['Prophet_pred_next_close_change_pct']


# Define evaluation metrics
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)





# Print evaluation metrics
print("\n===== PROPHET REGRESSION METRICS =====")

print(f"MAE:    {mae:.4f}")
print(f"MSE:    {mse:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MedAE:  {medae:.4f}")
print(f"R²:     {r2:.4f}")

print("\n===== RESULTS CREATED =====")
print(f"results_df shape: {results_df.shape}")

In [ ]:
results_df = test_df.copy()

# Add predictions
results_df['pred_next_close_change_pct'] = (y_test_pred).round(2)

# Impute error
results_df['pred_error'] = results_df['pred_next_close_change_pct'] - results_df['next_close_change_pct']

# Absolute error
results_df['abs_pred_error'] = results_df['pred_error'].abs()

# Directionally correct
results_df['directionally_correct'] = (
    np.sign(results_df['next_close_change_pct']) ==
    np.sign(results_df['pred_next_close_change_pct'])
).astype(int)

results_df